# Vergleich der Metrik-Modelle (Klassifikatoren und Regressoren)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook vergleicht direkt:
- **MixUp Regressor (BiLSTM)**
- **Synthetischer Regressor (BiLSTM)**

Wir evaluieren beide Modelle auf dem realen Lebenshilfe-Datensatz (`data/lebenshilfe/lebenshilfe_dataset_clean.json`), visualisieren die Vorhersagen über KDE-Dichteplots, vergleichen sie tabellarisch und prüfen die Korrelation zwischen beiden Modellen.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd(): break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


In [ ]:
nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


## 1. Modell-Klassen und Lade-Funktionen


In [ ]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out)

class DynamicVocab:
    def __init__(self, stoi_dict):
        self.stoi = stoi_dict
    def encode(self, tokens):
        return [self.stoi.get(t, self.stoi.get("<unk>", 1)) for t in tokens]


## 2. Alle Modelle und Vokabulare laden


In [ ]:
# MixUp Regressor
mixup_vocab_path = "data/vocabs/mixup_vocab.json"
mixup_model_path = "results/models/bilstm_mixup_regression.pt"

with open(mixup_vocab_path, "r", encoding="utf-8") as f:
    mixup_stoi = json.load(f)
mixup_vocab = DynamicVocab(mixup_stoi)
mixup_model = BiLSTMRegressor(len(mixup_stoi)).to(DEVICE)
mixup_model.load_state_dict(torch.load(mixup_model_path, map_location=DEVICE))
mixup_model.eval()

# Synthetic Regressor
synth_vocab_path = "data/vocabs/synthetic_vocab.json"
synth_model_path = "results/models/bilstm_synthetic_regression.pt"

with open(synth_vocab_path, "r", encoding="utf-8") as f:
    synth_stoi = json.load(f)
synth_vocab = DynamicVocab(synth_stoi)
synth_model = BiLSTMRegressor(len(synth_stoi)).to(DEVICE)
synth_model.load_state_dict(torch.load(synth_model_path, map_location=DEVICE))
synth_model.eval()
print("Regressoren erfolgreich geladen!")


## 3. Lebenshilfe-Datensatz laden & Vorhersagen berechnen


In [ ]:
LH_PATH = "data/lebenshilfe/lebenshilfe_dataset_clean.json"
with open(LH_PATH, "r", encoding="utf-8") as f:
    lh_data = json.load(f)
print(f"Geladene Artikelpaare aus Lebenshilfe: {len(lh_data)}")

def predict_batch_score(model, vocab, texts):
    scores = []
    for text in texts:
        tokens = [t.text.lower() for t in nlp(str(text)) if not t.is_space]
        encoded = vocab.encode(tokens)[:256]
        if not encoded:
            encoded = [vocab.stoi.get("<unk>", 1)]
        padded = encoded + [0] * (256 - len(encoded))
        inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            score = model(inp).squeeze().item()
        scores.append(score)
    return scores

as_texts = [item["as_text"] for item in lh_data]
ls_texts = [item["ls_text"] for item in lh_data]

print("Berechne Scores auf Lebenshilfe-Datensatz...")
as_scores_mixup = predict_batch_score(mixup_model, mixup_vocab, as_texts)
ls_scores_mixup = predict_batch_score(mixup_model, mixup_vocab, ls_texts)

as_scores_synth = predict_batch_score(synth_model, synth_vocab, as_texts)
ls_scores_synth = predict_batch_score(synth_model, synth_vocab, ls_texts)
print("Scores erfolgreich berechnet!")


## 4. Statistische Auswertung & KDE Density Plots
Hier vergleichen wir die Verteilungen der vorhergesagten Scores für Alltagssprache (AS) und Leichte Sprache (LS) beider Regressoren.


In [ ]:
print("=== MixUp Regressor Statistiken ===")
print(f"AS Mean Score: {np.mean(as_scores_mixup):.4f} +/- {np.std(as_scores_mixup):.4f}")
print(f"LS Mean Score: {np.mean(ls_scores_mixup):.4f} +/- {np.std(ls_scores_mixup):.4f}")
print()
print("=== Synthetic Regressor Statistiken ===")
print(f"AS Mean Score: {np.mean(as_scores_synth):.4f} +/- {np.std(as_scores_synth):.4f}")
print(f"LS Mean Score: {np.mean(ls_scores_synth):.4f} +/- {np.std(ls_scores_synth):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# MixUp Plot
sns.kdeplot(as_scores_mixup, fill=True, color="blue", label="AS (Alltagssprache)", ax=axes[0], clip=(0.0, 1.0))
sns.kdeplot(ls_scores_mixup, fill=True, color="green", label="LS (Leichte Sprache)", ax=axes[0], clip=(0.0, 1.0))
axes[0].set_title("MixUp Regressor - Vorhersagenverteilung")
axes[0].set_xlabel("Einfachheits-Score (Simplicity)")
axes[0].set_ylabel("Dichte (Density)")
axes[0].legend()

# Synthetic Plot
sns.kdeplot(as_scores_synth, fill=True, color="blue", label="AS (Alltagssprache)", ax=axes[1], clip=(0.0, 1.0))
sns.kdeplot(ls_scores_synth, fill=True, color="green", label="LS (Leichte Sprache)", ax=axes[1], clip=(0.0, 1.0))
axes[1].set_title("Synthetic Regressor - Vorhersagenverteilung")
axes[1].set_xlabel("Einfachheits-Score (Simplicity)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Korrelation und Tabelle auf dem Lebenshilfe-Datensatz


In [ ]:
all_mixup = as_scores_mixup + ls_scores_mixup
all_synth = as_scores_synth + ls_scores_synth
p_corr, _ = pearsonr(all_mixup, all_synth)
s_corr, _ = spearmanr(all_mixup, all_synth)
print(f"Pearson Korrelation zwischen MixUp und Synthetic:  {p_corr:.4f}")
print(f"Spearman Korrelation zwischen MixUp und Synthetic: {s_corr:.4f}")


In [ ]:
# Vergleichstabelle für einige zufällige Artikel erstellen
df_comp = pd.DataFrame({
    "Quelle / Dokument": [item.get("ls_filename", "lh").replace(".docx", "") for item in lh_data],
    "AS Text (Auszug)": [text[:80] + "..." for text in as_texts],
    "MixUp AS Score": as_scores_mixup,
    "Synth AS Score": as_scores_synth,
    "LS Text (Auszug)": [text[:80] + "..." for text in ls_texts],
    "MixUp LS Score": ls_scores_mixup,
    "Synth LS Score": ls_scores_synth
})
pd.set_option('display.max_columns', None)
display(df_comp.head(10))


## 6. Direkter Vergleich auf Beispielsätzen (Manuell)


In [ ]:
def compare_sentences(sentence):
    tokens = [t.text.lower() for t in nlp(sentence) if not t.is_space]
    mixup_enc = mixup_vocab.encode(tokens)[:256]
    mixup_inp = torch.tensor([mixup_enc], dtype=torch.long).to(DEVICE)
    
    synth_enc = synth_vocab.encode(tokens)[:256]
    synth_inp = torch.tensor([synth_enc], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        mixup_score = mixup_model(mixup_inp).squeeze().item()
        synth_score = synth_model(synth_inp).squeeze().item()
        
    print(f"Satz: '{sentence}'")
    print(f"-> MixUp Simplicity Score:    {mixup_score:.4f}")
    print(f"-> Synthetic Simplicity Score: {synth_score:.4f}")
    print("-" * 50)

sents = [
    "Wir heben das Geld ab.",
    "Es ist von fundamentaler Wichtigkeit, sämtliche Vorgänge präzise zu protokollieren.",
    "Wir schreiben alles genau auf.",
    "Die Ministerin für Arbeit und Soziales reiste gestern nach Frankreich, um neue Richtlinien zu besprechen.",
    "Die Ministerin reiste nach Frankreich."
]

for s in sents:
    compare_sentences(s)


## 7. Eigene Sätze eingeben


In [ ]:
custom_sentence = "Geben Sie hier Ihren eigenen Satz ein..."
compare_sentences(custom_sentence)
